In [ ]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
import numpy as np

In [ ]:
#Load Dataset
df = pd.read_csv("../DataSet/mushroom.csv")
df.head()

#Check for null
# df.isna().sum()

In [ ]:
df.info()

In [ ]:
def distance(x, y):
    return np.sqrt(np.sum((x - y) ** 2))

def kmeans(df, k=4, tol=0.05):
    centroids = df.sample(n=k)
    prev_error = float('inf')
    
    while True:
        clusters = []
        for i, row in df.iterrows():
            dists = [distance(row, c) for c in centroids.values]
            clusters.append(dists.index(min(dists)))
        
        clusters = np.array(clusters)    
        new_centroids = pd.DataFrame([df[clusters==i].mean() for i in range(k)]) 
        
        error = np.mean([distance(df.iloc[i], new_centroids.iloc[clusters[i]])**2
                         for i in range(len(df))])
                         
        if abs(prev_error - error) < tol:
            break
            
        prev_error = error        
        centroids = new_centroids
        
    return centroids, clusters, error

In [ ]:
centroids_list = []
clusters_list = []
errors = []

for i in range(20):
    centroids, clusters, error = kmeans(df[['stem-height','Attack','Defense']], k=2)
    
    centroids_list.append(centroids)
    clusters_list.append(clusters) 
    errors.append(error)
    
    print(f"Run {i+1} - Error: {error:.2f}")

best_run = np.argmin(errors) 

fig, ax = plt.subplots()
ax.scatter(df['Attack'], df['Defense'], c=clusters_list[best_run])
ax.scatter(centroids_list[best_run]['Attack'], 
           centroids_list[best_run]['Defense'],
           c='red', marker='+')
ax.set_xlabel('Attack')
ax.set_ylabel('Defense')
ax.set_title(f'Best: Run {best_run+1} Clustering')

plt.show()
